In [1]:
import pandas as pd

In [2]:
from pathlib import Path

raw_path = Path("../data/raw")

csv_files = sorted(raw_path.glob("*.csv"))

for file in csv_files:
    print(file.name)

Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Friday-WorkingHours-Morning.pcap_ISCX.csv
Monday-WorkingHours.pcap_ISCX.csv
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Tuesday-WorkingHours.pcap_ISCX.csv
Wednesday-workingHours.pcap_ISCX.csv


In [19]:
for file in csv_files:
    temp_df = pd.read_csv(file, nrows=5)
    temp_df.columns = temp_df.columns.str.strip()
    
    print(file.name, "→", len(temp_df.columns), "columns")

Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv → 79 columns
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv → 79 columns
Friday-WorkingHours-Morning.pcap_ISCX.csv → 79 columns
Monday-WorkingHours.pcap_ISCX.csv → 79 columns
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv → 79 columns
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv → 79 columns
Tuesday-WorkingHours.pcap_ISCX.csv → 79 columns
Wednesday-workingHours.pcap_ISCX.csv → 79 columns


In [20]:
dataframes = []

for file in csv_files:
    temp_df = pd.read_csv(file)
    temp_df.columns = temp_df.columns.str.strip()
    dataframes.append(temp_df)

df = pd.concat(dataframes, ignore_index=True)

print("Combined dataset shape:", df.shape)

Combined dataset shape: (2830743, 79)


In [21]:
print(df["Label"].value_counts())

Label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64


In [22]:
df["Target"] = df["Label"].apply(lambda x: 0 if x == "BENIGN" else 1)

print(df["Target"].value_counts())

Target
0    2273097
1     557646
Name: count, dtype: int64


In [23]:
import numpy as np

print("Infinite values:", np.isinf(df.select_dtypes(include="number")).sum().sum())

Infinite values: 4376


In [24]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)

In [25]:
print("Total missing values:", df.isnull().sum().sum())

Total missing values: 5734


In [26]:
missing = df.isnull().sum()

print(missing[missing > 0])

Flow Bytes/s      2867
Flow Packets/s    2867
dtype: int64


In [27]:
print("Flow Bytes/s median:", df["Flow Bytes/s"].median())
print("Flow Packets/s median:", df["Flow Packets/s"].median())

Flow Bytes/s median: 4586.600756
Flow Packets/s median: 109.4759992228


In [28]:
df["Flow Bytes/s"] = df["Flow Bytes/s"].fillna(df["Flow Bytes/s"].median())
df["Flow Packets/s"] = df["Flow Packets/s"].fillna(df["Flow Packets/s"].median())

In [29]:
print("Total missing values:", df.isnull().sum().sum())

Total missing values: 0


In [30]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 308381


In [31]:
duplicate_rows = df[df.duplicated(keep=False)]

print(duplicate_rows["Target"].value_counts())

Target
0    236257
1    167293
Name: count, dtype: int64


In [32]:
df = df.drop_duplicates().reset_index(drop=True)

In [33]:
print("New dataset shape:", df.shape)
print("Duplicate rows remaining:", df.duplicated().sum())

New dataset shape: (2522362, 80)
Duplicate rows remaining: 0


In [34]:
print(df.dtypes.value_counts())

int64      55
float64    24
object      1
Name: count, dtype: int64


In [35]:
print(df.columns.tolist())

['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWE Flag Count', 'ECE Flag Count

In [36]:
constant_columns = df.nunique()

print(constant_columns[constant_columns == 1])

Bwd PSH Flags           1
Bwd URG Flags           1
Fwd Avg Bytes/Bulk      1
Fwd Avg Packets/Bulk    1
Fwd Avg Bulk Rate       1
Bwd Avg Bytes/Bulk      1
Bwd Avg Packets/Bulk    1
Bwd Avg Bulk Rate       1
dtype: int64


In [38]:
print(df.describe().T[["min", "max"]])

                              min          max
Destination Port              0.0      65535.0
Flow Duration               -13.0  119999998.0
Total Fwd Packets             1.0     219759.0
Total Backward Packets        0.0     291922.0
Total Length of Fwd Packets   0.0   12900000.0
...                           ...          ...
Idle Mean                     0.0  120000000.0
Idle Std                      0.0   76900000.0
Idle Max                      0.0  120000000.0
Idle Min                      0.0  120000000.0
Target                        0.0          1.0

[71 rows x 2 columns]


In [39]:
X = df.drop(columns=["Label", "Target"])
y = df["Target"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (2522362, 70)
y shape: (2522362,)


In [40]:
print("Missing values in X:", X.isnull().sum().sum())
print("Infinite values in X:", np.isinf(X).sum().sum())

Missing values in X: 0
Infinite values in X: 0


In [41]:
print(df["Target"].value_counts())

Target
0    2096484
1     425878
Name: count, dtype: int64


In [42]:
dataframes = []

for file in csv_files:
    temp_df = pd.read_csv(file)
    temp_df.columns = temp_df.columns.str.strip()
    
    temp_df["Day"] = file.stem
    
    dataframes.append(temp_df)

df_split = pd.concat(dataframes, ignore_index=True)

print("Shape:", df_split.shape)
print(df_split["Day"].value_counts())

Shape: (2830743, 80)
Day
Wednesday-workingHours.pcap_ISCX                           692703
Monday-WorkingHours.pcap_ISCX                              529918
Tuesday-WorkingHours.pcap_ISCX                             445909
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX    288602
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX           286467
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX               225745
Friday-WorkingHours-Morning.pcap_ISCX                      191033
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX         170366
Name: count, dtype: int64


In [43]:
print(df_split["Day"].unique())

['Friday-WorkingHours-Afternoon-DDos.pcap_ISCX'
 'Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX'
 'Friday-WorkingHours-Morning.pcap_ISCX' 'Monday-WorkingHours.pcap_ISCX'
 'Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX'
 'Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX'
 'Tuesday-WorkingHours.pcap_ISCX' 'Wednesday-workingHours.pcap_ISCX']


In [44]:
df_split["Target"] = df_split["Label"].apply(
    lambda x: 0 if x == "BENIGN" else 1
)

print(df_split["Target"].value_counts())

Target
0    2273097
1     557646
Name: count, dtype: int64


In [45]:
import numpy as np

df_split.replace([np.inf, -np.inf], np.nan, inplace=True)

print("Total missing values:", df_split.isnull().sum().sum())

Total missing values: 5734


In [46]:
missing = df_split.isnull().sum()

print(missing[missing > 0])

Flow Bytes/s      2867
Flow Packets/s    2867
dtype: int64


In [47]:
df_split["Flow Bytes/s"] = df_split["Flow Bytes/s"].fillna(
    df_split["Flow Bytes/s"].median()
)

df_split["Flow Packets/s"] = df_split["Flow Packets/s"].fillna(
    df_split["Flow Packets/s"].median()
)

print("Total missing values:", df_split.isnull().sum().sum())

Total missing values: 0


In [48]:
print("Duplicate rows:", df_split.duplicated().sum())

Duplicate rows: 256479


In [49]:
df_split = df_split.drop_duplicates().reset_index(drop=True)

print("New dataset shape:", df_split.shape)
print("Duplicate rows remaining:", df_split.duplicated().sum())

New dataset shape: (2574264, 81)
Duplicate rows remaining: 0


In [50]:
constant_columns = df_split.nunique()

print(constant_columns[constant_columns == 1])

Bwd PSH Flags           1
Bwd URG Flags           1
Fwd Avg Bytes/Bulk      1
Fwd Avg Packets/Bulk    1
Fwd Avg Bulk Rate       1
Bwd Avg Bytes/Bulk      1
Bwd Avg Packets/Bulk    1
Bwd Avg Bulk Rate       1
dtype: int64


In [51]:
constant_columns = df_split.nunique()
constant_columns = constant_columns[constant_columns == 1].index

df_split = df_split.drop(columns=constant_columns)

print("Removed constant columns:", list(constant_columns))
print("New dataset shape:", df_split.shape)

Removed constant columns: ['Bwd PSH Flags', 'Bwd URG Flags', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']
New dataset shape: (2574264, 73)


In [52]:
print(df_split["Target"].value_counts())

Target
0    2148386
1     425878
Name: count, dtype: int64


In [53]:
print(df_split.groupby("Day")["Target"].value_counts())

Day                                                      Target
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX             1         128016
                                                         0          95096
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX         0         123295
                                                         1          90819
Friday-WorkingHours-Morning.pcap_ISCX                    0         182192
                                                         1           1953
Monday-WorkingHours.pcap_ISCX                            0         502983
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX  0         252936
                                                         1             36
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX       0         162157
                                                         1           2143
Tuesday-WorkingHours.pcap_ISCX                           0         412692
                                                

In [54]:
train_days = [
    "Monday-WorkingHours.pcap_ISCX",
    "Tuesday-WorkingHours.pcap_ISCX",
    "Wednesday-workingHours.pcap_ISCX",
    "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX",
    "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX"
]

test_days = [
    "Friday-WorkingHours-Morning.pcap_ISCX",
    "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX",
    "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX"
]

train_df = df_split[df_split["Day"].isin(train_days)].copy()
test_df = df_split[df_split["Day"].isin(test_days)].copy()

print("Training shape:", train_df.shape)
print("Testing shape:", test_df.shape)

Training shape: (1952893, 73)
Testing shape: (621371, 73)


In [55]:
X_train = train_df.drop(columns=["Label", "Target", "Day"])
y_train = train_df["Target"]

X_test = test_df.drop(columns=["Label", "Target", "Day"])
y_test = test_df["Target"]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (1952893, 70)
y_train: (1952893,)
X_test: (621371, 70)
y_test: (621371,)


In [56]:
print("Training class distribution:")
print(y_train.value_counts())

print("\nTesting class distribution:")
print(y_test.value_counts())

Training class distribution:
Target
0    1747803
1     205090
Name: count, dtype: int64

Testing class distribution:
Target
0    400583
1    220788
Name: count, dtype: int64


In [57]:
print("Missing values in X_train:", X_train.isnull().sum().sum())
print("Infinite values in X_train:", np.isinf(X_train).sum().sum())

print("Missing values in X_test:", X_test.isnull().sum().sum())
print("Infinite values in X_test:", np.isinf(X_test).sum().sum())

Missing values in X_train: 0
Infinite values in X_train: 0
Missing values in X_test: 0
Infinite values in X_test: 0


In [58]:
from pathlib import Path

processed_path = Path("../data/processed")
processed_path.mkdir(parents=True, exist_ok=True)

X_train.to_csv(processed_path / "X_train.csv", index=False)
y_train.to_csv(processed_path / "y_train.csv", index=False)

X_test.to_csv(processed_path / "X_test.csv", index=False)
y_test.to_csv(processed_path / "y_test.csv", index=False)

print("Train and test data saved successfully.")

Train and test data saved successfully.
